# 800 — CPTAC-3 transcriptomic input readiness

## Objective

Evaluate the integrity and analytical compatibility of an external CPTAC-3 RNA-seq source cohort against the **frozen Phase 4 transcriptomic reference**.

The source inventory contains **1,082** structurally simple primary-tumor cases. The proposed external cohort contains **721** case–file pairs distributed across seven harmonized lineages. These populations answer different questions and remain distinct throughout the notebook.

**Scope:** validate upstream source identities, full-source feature coverage, the proposed cohort handoff, lineage composition and outstanding requirements for analysis. No source case is selected here, and no program projection, replication test, clinical prediction or causal inference is performed.

**Inputs:** audited Notebooks 112 and 113, their SHA-256 manifests, local STAR Count files and frozen Phase 4 artifacts. Only the feature-coverage audit reads raw STAR files, and only when its verified derived cache is absent.


In [ ]:
# =============================================================================
# Imports, paths and exact-file utilities
# =============================================================================

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths

ROOT = Paths.root
META = Paths.metadata
REGISTRY_PATH = ROOT / "config" / "artifact_registry.json"
MANIFEST_PATH = META / "113_source_provenance_manifest.json"
READINESS_PATH = META / "113_cohort_readiness.json"

COHORT_NAME = "113_proposed_external_source_cohort.csv"
LEDGER_NAME = "113_primary_candidate_case_eligibility.csv"
LINEAGE_NAME = "113_proposed_source_cohort_lineage_counts.csv"
ATTRITION_NAME = "113_source_handoff_attrition.csv"
G6_NAME = "113_g6_raw_count_compatibility.csv"
G8_COVARIATES_NAME = "113_g8_covariate_coverage.csv"
G8_WORKFLOW_NAME = "113_g8_workflow_lineage_support.csv"
G8_CONTEXT_NAME = "113_g8_source_lineage_support.csv"

EXPECTED_LINEAGES = {
    "TCGA-HNSC": 36,
    "TCGA-LUAD": 178,
    "TCGA-LUSC": 89,
    "TCGA-PAAD": 82,
    "TCGA-SKCM": 138,
    "TCGA-STAD": 142,
    "TCGA-UCEC": 56,
}


In [ ]:
# =============================================================================
# Project-relative input and schema utilities
# =============================================================================

def required_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Required local input missing: {path}. "
            "Use the existing executed Notebook 113 outputs; do not reconstruct the cohort in 800."
        )
    return path

def read_json(path):
    return json.loads(required_file(path).read_text(encoding="utf-8"))

def file_sha(path):
    return calculate_sha256(required_file(path))

def read_csv(name):
    return pd.read_csv(required_file(META / name), dtype="string")

def require_columns(frame, columns, label):
    absent = set(columns) - set(frame.columns)
    if absent:
        raise ValueError(f"{label}: missing columns {sorted(absent)}")

def check(condition, message):
    if not bool(condition):
        raise ValueError(message)

print("Repository:", ROOT)
print("Mode: offline source-handoff verification; program scoring disabled")


## 1. Source manifest and evidence integrity

Reconcile the Notebook 113 source manifest with its readiness record and the persisted upstream evidence. A mismatch in a source identity or fingerprint blocks analysis; it does not justify rebuilding or replacing the proposed cohort.


In [ ]:
# =============================================================================
# Verify source manifest and readiness, including all recorded SHA-256 values
# =============================================================================

manifest = read_json(MANIFEST_PATH)
readiness = read_json(READINESS_PATH)
check(manifest.get("notebook") == 113, "Unexpected source manifest")
check(readiness.get("notebook") == 113, "Unexpected source readiness")
check(readiness.get("manifest_sha256") == file_sha(MANIFEST_PATH),
      "113 readiness does not match the source manifest")
check(manifest.get("cohort_frozen") is False, "113 manifest freeze state changed")
check(readiness.get("cohort_frozen") is False, "113 cohort is unexpectedly frozen")
check(readiness.get("external_analysis_authorized") is False,
      "Unexpected upstream biological-analysis authorization")

for relative, expected_sha in manifest["input_sha256"].items():
    check(file_sha(ROOT / relative) == expected_sha,
          f"113 source input fingerprint mismatch: {relative}")

for label, item in manifest.get("reviewed_evidence", {}).items():
    if isinstance(item, dict) and "path" in item and "sha256" in item:
        check(file_sha(ROOT / item["path"]) == item["sha256"],
              f"113 reviewed-source fingerprint mismatch: {label}")


In [ ]:
# =============================================================================
# Verify source output fingerprints and readiness
# =============================================================================

output_info = manifest["outputs"]
for name, info in output_info.items():
    path = META / name
    check(file_sha(path) == info["sha256"],
          f"113 output fingerprint mismatch: {name}")

check(readiness.get("proposed_source_cohort_sha256") ==
      output_info[COHORT_NAME]["sha256"],
      "113 readiness/cohort manifest fingerprints disagree")

print("113 source manifest: VERIFIED")
print("113 source inputs: VERIFIED", len(manifest["input_sha256"]))
print("113 output files: VERIFIED", len(output_info))
print("113 readiness: source cohort proposed; external analysis not authorized")


## 2. Candidate cohort and lineage membership

Verify the one-case–one-file unit and the predeclared intersection of G3 external source provenance, G4 lineage mapping and G6 input compatibility. The **721 proposed cases** are a subset of the **1,082 structural candidates**; the 288 multiply represented primary cases are not arbitrarily resolved.

Lineage counts are reported separately. Additional RNA files with ambiguous ancestry remain sensitivity metadata rather than automatic exclusions.


In [ ]:
# =============================================================================
# Validate 1,082-case ledger, 721-case handoff and exact inclusion intersection
# =============================================================================

cohort = read_csv(COHORT_NAME)
ledger = read_csv(LEDGER_NAME)
lineage_file = read_csv(LINEAGE_NAME)
attrition = read_csv(ATTRITION_NAME)

identity = ["case_uuid", "file_id", "sample_uuid", "aliquot_uuid"]
require_columns(cohort, identity + [
    "tcga_lineage", "g3_class", "g4_status", "g6_matrix_status",
    "source_handoff_status", "qc_status", "other_ambiguous_rna_flag",
], "113 cohort")
require_columns(ledger, identity + [
    "tcga_lineage", "g3_class", "g4_status", "g6_matrix_status",
    "source_handoff_status", "other_ambiguous_rna_flag",
], "113 complete case ledger")

check(len(ledger) == 1082 and ledger["case_uuid"].is_unique,
      "G2 1,082-case candidate ledger mismatch")
check(len(cohort) == 721, "Proposed source cohort is not 721 rows")
for label, frame in [("source cohort", cohort), ("G2 ledger", ledger)]:
    check(frame[identity].notna().all().all(), f"{label}: missing identity")
    check(frame["case_uuid"].is_unique and frame["file_id"].is_unique,
          f"{label}: case/file duplicates")

selected = (
    ledger["g3_class"].eq("COHORT_SUPPORTED_EXTERNAL")
    & ledger["g4_status"].eq("ACCEPTED")
    & ledger["g6_matrix_status"].eq("COMPATIBLE_RAW_COUNTS")
)
check(selected.notna().all(), "Incomplete upstream selection labels")
check(int(selected.sum()) == 721, "G3/G4/G6 intersection drift")
check(set(cohort["case_uuid"]) == set(ledger.loc[selected, "case_uuid"]),
      "113 cohort differs from the reviewed eligibility intersection")


In [ ]:
# Apply outcome-blind candidate intersection

included = ledger.loc[selected, ["case_uuid"] + identity[1:] + [
    "tcga_lineage", "g3_class", "g4_status", "g6_matrix_status"
]]
comparison = cohort[included.columns].merge(
    included, on="case_uuid", validate="one_to_one", suffixes=("_cohort", "_ledger")
)
check(len(comparison) == 721, "Membership case merge incomplete")
for column in included.columns.drop("case_uuid"):
    check(
        comparison[f"{column}_cohort"].eq(
            comparison[f"{column}_ledger"]
        ).fillna(False).all(),
        f"113 cohort and ledger disagree in {column}",
    )


In [ ]:
# Cross-check exact case and file identity

check(cohort["source_handoff_status"].eq("PROPOSED_SOURCE_COHORT").all(),
      "Unexpected 113 source membership status")
check(cohort["g3_class"].eq("COHORT_SUPPORTED_EXTERNAL").all(),
      "Cohort has unresolved G3 status")
check(cohort["g4_status"].eq("ACCEPTED").all(),
      "Cohort has unaccepted G4 lineage")
check(cohort["g6_matrix_status"].eq("COMPATIBLE_RAW_COUNTS").all(),
      "Cohort has incompatible raw counts")

check(int(ledger["other_ambiguous_rna_flag"].str.lower().eq("true").sum()) == 37,
      "G2 additional ambiguous RNA source flags changed")
check(int(manifest["source_counts"]["multiplicity_cases"]) == 288,
      "Upstream G2 multiplicity boundary changed")
check(int(manifest["source_counts"]["g2_simple_cases"]) == 1082,
      "Upstream G2 source candidate boundary changed")

print("Identity and exact source membership: VERIFIED")
print("G2 source candidates:", len(ledger), "| proposed cases:", len(cohort))
print("G3 residual donor-level independence uncertainty remains; QC is not certified")


In [ ]:
# =============================================================================
# Verify all seven source-to-TCGA lineage denominators and 113 attrition
# =============================================================================

require_columns(lineage_file, ["tcga_lineage", "n_cases"], "113 lineage counts")
require_columns(attrition, ["source_handoff_status", "n_cases"], "113 attrition")

observed = cohort.groupby("tcga_lineage")["case_uuid"].nunique().to_dict()
check(observed == EXPECTED_LINEAGES, f"Unexpected proposed lineage counts: {observed}")

reported = dict(zip(
    lineage_file["tcga_lineage"],
    pd.to_numeric(lineage_file["n_cases"], errors="raise").astype(int),
))
check(reported == EXPECTED_LINEAGES, "Source lineage summary differs from cohort")
check(pd.to_numeric(attrition["n_cases"], errors="raise").sum() == 1082,
      "113 case-attrition summary does not reconcile to G2")

lineage_support = (
    cohort.groupby("tcga_lineage", as_index=False)
    .agg(n_cases=("case_uuid", "nunique"),
         flagged_additional_ambiguous=(
             "other_ambiguous_rna_flag",
             lambda values: int(values.str.lower().eq("true").sum()),
         ))
    .sort_values("tcga_lineage")
    .reset_index(drop=True)
)
display(lineage_support)
print("G2 to proposed source cohort attrition:", len(ledger) - len(cohort))
print("All seven lineage denominators: VERIFIED; no lineage is yet inferentially approved")


## 3. Source provenance and QC evidence

The GDC and PDC provenance evidence supports external-cohort use at the **study-design level**, but does not establish genetic independence for every donor.

Provider MD5 and STAR count-vector checks establish source integrity and basic input compatibility, not independent sequencing or library-quality approval. Unavailable QC evidence is reported as unassessed.


In [ ]:
# =============================================================================
# Report source provenance and QC without overclaiming donor/QC certification
# =============================================================================

require_columns(ledger, ["g3_class", "qc_status"], "113 ledger")
g3_counts = ledger["g3_class"].value_counts(dropna=False)
qc_counts = cohort["qc_status"].value_counts(dropna=False)

for label, expected in readiness.get("g3_class_counts", {}).items():
    check(int(g3_counts.get(label, 0)) == int(expected),
          f"G3 readiness count mismatch: {label}")
for label, expected in readiness.get("qc_status_counts", {}).items():
    check(int(ledger["qc_status"].eq(label).sum()) == int(expected),
          f"G5 readiness count mismatch: {label}")

display(pd.DataFrame({
    "g3_class": g3_counts.index.astype(str),
    "n_g2_cases": g3_counts.to_numpy(),
}))
display(qc_counts.rename_axis("cohort_qc_status").reset_index(name="n_cases"))
print("G3: cohort-level externality supported; individual donor overlap not genetically excluded")
print("G5: missing read-level QC remains QC_UNASSESSED, not QC_PASS")


## 4. Frozen reference and candidate-file compatibility

Verify the immutable Phase 4 gene universe and three transcriptomic program artifacts. Inspect the separately audited G6 results for the 721 proposed sources.

The resulting checks concern **feature and raw-count availability**, not normalization across platforms, program activity or external replication.


In [ ]:
# =============================================================================
# Verify frozen Phase 4 reference artifacts and 113 raw-count compatibility
# =============================================================================

registry = read_json(REGISTRY_PATH)
artifacts = registry["artifacts"]
reference_keys = {
    "shared_genes": "phase4.400.cross_system_shared_gene_universe",
    "program_catalog": "phase4.401.consensus_transcriptomic_program_catalog",
    "program_weights": "phase4.401.consensus_transcriptomic_gene_weights",
}

references = {}
for label, key in reference_keys.items():
    entry = artifacts[key]
    check(entry["status"] == "frozen", f"Phase 4 reference not frozen: {key}")
    target = ROOT / entry["path"]
    check(file_sha(target) == entry["sha256"],
          f"Frozen reference fingerprint mismatch: {key}")
    references[label] = pd.read_csv(target, dtype="string")
    check(len(references[label]) == entry["shape"][0],
          f"Frozen reference row-count mismatch: {key}")

gene_frame = references["shared_genes"]
require_columns(gene_frame, ["tumor_gene_id"], "Phase 4 gene universe")
gene_ids = gene_frame["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
check(len(gene_ids) == 2389 and gene_ids.notna().all() and gene_ids.is_unique,
      "Expected 2,389 unique frozen Ensembl features")
check(len(references["program_catalog"]) == 3, "Expected three frozen programs")
check(len(references["program_weights"]) == 3 * 2389,
      "Expected three complete 2,389-row loading vectors")


In [ ]:
# =============================================================================
# Verify candidate-file matrix availability
# =============================================================================

g6 = read_csv(G6_NAME)
require_columns(g6, [
    "case_uuid", "file_id", "status", "n_shared_genes_found",
    "n_duplicated_gene_ids", "has_unstranded",
], "113 G6 matrix audit")
check(g6["case_uuid"].is_unique and g6["file_id"].is_unique,
      "113 G6 matrix audit includes duplicated case/file identities")

g6_selected = g6.set_index("case_uuid").reindex(cohort["case_uuid"])
check(g6_selected["status"].eq("COMPATIBLE_RAW_COUNTS").all(),
      "One or more proposed files failed the 113 raw-count audit")
check(
    pd.to_numeric(g6_selected["n_shared_genes_found"], errors="raise").eq(2389).all(),
    "A proposed file lacks frozen genes",
)
check(
    pd.to_numeric(g6_selected["n_duplicated_gene_ids"], errors="raise").eq(0).all(),
    "A proposed file has duplicated version-normalized gene IDs",
)
check(g6_selected["has_unstranded"].str.lower().eq("true").all(),
      "A proposed file lacks the audited unstranded count field")
check(g6_selected["file_id"].to_numpy().tolist() == cohort["file_id"].tolist(),
      "G6 source file IDs differ from the proposed cohort")

print("Phase 4 frozen inputs: VERIFIED — three programs, 2,389 genes")
print("G6: 721 source raw-count files COMPATIBLE (113 evidence reused)")
print("Separate-cohort normalization and frozen-loading projection: NOT YET SPECIFIED")


## 5. Full-source frozen-feature coverage

Evaluate **all 1,082** uniquely mapped primary STAR Count files against the **2,389 exact Phase 4 Ensembl identifiers**.

For each source file, record the exact identifiers present, missing exact identifiers, version-insensitive diagnostic matches and detected frozen-gene counts. Aggregate exact feature presence across files. Gene-ID coverage and nonnegative unstranded counts are input-compatibility observations; they do not establish independent RNA-seq QC.

A deterministic derived cache under `data/interim/metadata/` is reusable only when the fingerprinted inputs, source identities, reference genes and output files are unchanged. Otherwise the analysis stops or requires an explicit full rescan.


In [ ]:
# =============================================================================
# Reconcile 1,082 exact source identities with the 112 technical audit
# =============================================================================
import hashlib

import numpy as np

TECH_112_PATH = META / "112_cptac3_rna_simple_primary_technical_inventory.csv"
HANDOFF_112_PATH = META / "112_cptac3_qc_provenance_handoff.json"
E_FILE_PATH = META / "800_cptac3_e_exact_feature_file_diagnostics.csv"
E_GENE_PATH = META / "800_cptac3_e_exact_frozen_gene_coverage.csv"
E_MANIFEST_PATH = META / "800_cptac3_e_exact_feature_manifest.json"
E_STAR_ROOT = ROOT / "data/raw/cptac3/rnaseq_star_counts"
E_STAR_SPECIAL = ("N_unmapped", "N_multimapping", "N_noFeature", "N_ambiguous")
E_AUDIT_SCHEMA = "phase8_E_full1082_exact_frozen_features_v1"
E_REBUILD_FULL_STAR_AUDIT = False  # Explicit opt-in to replace a valid cache

source_112_manifest = read_json(HANDOFF_112_PATH)
receipt_112 = (source_112_manifest.get("outputs") or {}).get("technical_inventory")
check(isinstance(receipt_112, dict), "112 handoff lacks technical inventory provenance")
check(receipt_112.get("sha256") == file_sha(TECH_112_PATH),
      "112 STAR technical inventory SHA-256 disagrees with its handoff")
check(receipt_112.get("rows") == 1082, "112 technical handoff row count changed")
check(receipt_112.get("path") ==
      str(TECH_112_PATH.relative_to(ROOT)).replace("\\", "/"),
      "112 handoff path is not the expected technical inventory")
source_112 = pd.read_csv(required_file(TECH_112_PATH), dtype="string")
source_identity = ["case_uuid", "file_id", "sample_uuid", "aliquot_uuid"]
require_columns(source_112, source_identity + [
    "n_gene_rows", "n_duplicate_versionless_ids", "n_star_special_rows",
    "n_invalid_unstranded_gene_values",
    "unstranded_nonnegative_integer_vector", "unstranded_gene_count_total",
    "n_detected_star_genes",
    *["star_" + k.lower() for k in E_STAR_SPECIAL],
], "112 full STAR technical inventory")
require_columns(ledger, source_identity + [
    "filename", "size_bytes", "provider_md5", "primary_site",
    "disease_type", "source_handoff_status",
], "113 full G2 source ledger")
check(len(source_112) == len(ledger) == 1082,
      "E full-source audit requires 1,082 distinct G2 files")
check(source_112["case_uuid"].is_unique and source_112["file_id"].is_unique,
      "112 input is not one sample and file per case")


In [ ]:
# =============================================================================
# Reconcile the full G2 source identity universe
# =============================================================================

source_identity_left = ledger[source_identity].sort_values(
    source_identity, kind="stable").reset_index(drop=True)
source_identity_right = source_112[source_identity].sort_values(
    source_identity, kind="stable").reset_index(drop=True)
check(source_identity_left.equals(source_identity_right),
      "112 and 113 source case/file/sample/aliquot identities disagree")


In [ ]:
# =============================================================================
# Define exact and version-independent frozen Ensembl IDs
# =============================================================================

e_frozen_exact = gene_frame["tumor_gene_id"].astype("string").reset_index(drop=True)
check(len(e_frozen_exact) == 2389
      and not e_frozen_exact.isna().any()
      and e_frozen_exact.is_unique, "Invalid original Phase 4 exact gene IDs")
e_frozen_bases = e_frozen_exact.str.replace(r"\.\d+$", "", regex=True)
check(e_frozen_bases.is_unique, "Frozen versionless Ensembl IDs collide")
e_frozen_set = set(e_frozen_exact.tolist())
e_frozen_base_set = set(e_frozen_bases.tolist())

e_source = ledger[source_identity + [
    "filename", "size_bytes", "provider_md5", "primary_site",
    "disease_type", "source_handoff_status",
]].sort_values(["case_uuid", "file_id"], kind="stable").reset_index(drop=True)


In [ ]:
# =============================================================================
# Fingerprint the source and reference used by the feature audit
# =============================================================================

E_INPUT_FINGERPRINTS = {
    "112_technical_inventory_sha256": file_sha(TECH_112_PATH),
    "112_handoff_sha256": file_sha(HANDOFF_112_PATH),
    "113_source_manifest_sha256": file_sha(MANIFEST_PATH),
    "113_g2_ledger_sha256": file_sha(META / LEDGER_NAME),
    "phase4_shared_gene_reference_sha256": file_sha(
        ROOT / artifacts[reference_keys["shared_genes"]]["path"]
    ),
    "111_primary_source_index_sha256": manifest["input_sha256"][
        str((META / "111_cptac3_provisional_primary_tumor_files.csv").relative_to(ROOT))
    ],
}
E_SOURCE_IDENTITY_SHA256 = hashlib.sha256(
    e_source[source_identity + ["filename", "size_bytes", "provider_md5"]]
    .to_csv(index=False, lineterminator="\n").encode("utf-8")
).hexdigest()
print("Full-source sample scope:", len(e_source), "source files")
print("Phase 4 exact/versionless reference genes:", len(e_frozen_exact))
print("112 and 113 case/file/sample/aliquot identities: EXACT MATCH")
print("Proposed source cohort is unchanged:", len(cohort), "cases")


In [ ]:
# =============================================================================
# Resolve the derived feature-coverage cache
# =============================================================================

e_cache_paths = [E_FILE_PATH, E_GENE_PATH, E_MANIFEST_PATH]
cache_exists = [path.is_file() for path in e_cache_paths]

if any(cache_exists) and not all(cache_exists) and not E_REBUILD_FULL_STAR_AUDIT:
    raise ValueError(
        "Incomplete feature-coverage cache. Rebuild only after review."
    )

E_SCAN_PERFORMED = E_REBUILD_FULL_STAR_AUDIT or not all(cache_exists)

if not E_SCAN_PERFORMED:
    cached_e_manifest = read_json(E_MANIFEST_PATH)
    check(cached_e_manifest.get("schema") == E_AUDIT_SCHEMA, "E schema changed")
    check(
        cached_e_manifest.get("verification_status")
        in (None, "PENDING_CROSSCHECK", "VERIFIED"),
        "Unexpected feature-coverage cache verification state",
    )
    check(
        cached_e_manifest.get("inputs") == E_INPUT_FINGERPRINTS,
        "Feature-coverage cache does not match its upstream inputs",
    )
    check(
        cached_e_manifest.get("source_identity_sha256") == E_SOURCE_IDENTITY_SHA256,
        "Cached G2 source file identity has changed",
    )
    check(
        cached_e_manifest.get("n_source_files") == 1082
        and cached_e_manifest.get("n_frozen_genes") == 2389,
        "Feature-coverage cache denominator changed",
    )
    check(
        file_sha(E_FILE_PATH) == cached_e_manifest["file_output_sha256"]
        and file_sha(E_GENE_PATH) == cached_e_manifest["gene_output_sha256"],
        "Cached feature-coverage output fingerprints disagree",
    )

print("Feature-coverage mode:", "STAR scan" if E_SCAN_PERFORMED else "cache")


In [ ]:
# =============================================================================
# Read one STAR source and preserve exact feature identifiers
# =============================================================================

def read_star_source(row):
    path = E_STAR_ROOT / str(row.file_id) / str(row.filename)
    if not path.is_file():
        raise FileNotFoundError(f"Expected STAR source not found: {path}")
    if path.stat().st_size != int(row.size_bytes):
        raise ValueError(f"GDC file size disagreement for {row.file_id}")

    table = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        usecols=["gene_id", "unstranded"],
        dtype={"gene_id": "string", "unstranded": "string"},
    )
    if table["gene_id"].isna().any():
        raise ValueError(f"Missing STAR gene IDs in {row.file_id}")

    special_mask = table["gene_id"].str.startswith("N_", na=False)
    genes = table.loc[~special_mask].copy()
    special = table.loc[special_mask].copy()

    if genes.empty or genes["gene_id"].duplicated().any():
        raise ValueError(f"Empty or duplicate exact gene IDs in {row.file_id}")
    return genes, special


In [ ]:
# =============================================================================
# Describe feature identity and count vectors for one STAR source
# =============================================================================

def inspect_star_features(genes):
    gene_ids = genes["gene_id"]
    versionless_ids = gene_ids.str.replace(r"\.\d+$", "", regex=True)
    exact_presence = e_frozen_exact.isin(set(gene_ids))
    versionless_presence = e_frozen_bases.isin(set(versionless_ids))

    return {
        "n_gene_rows": len(gene_ids),
        "n_duplicate_versionless_ids": int(
            versionless_ids.value_counts(dropna=False).gt(1).sum()
        ),
        "n_frozen_exact_present": int(exact_presence.sum()),
        "n_frozen_exact_missing": len(e_frozen_exact) - int(exact_presence.sum()),
        "n_frozen_versionless_diagnostic": int(versionless_presence.sum()),
        "n_frozen_versionless_only_diagnostic": int(
            (versionless_presence & ~exact_presence).sum()
        ),
    }, exact_presence.to_numpy(dtype=np.int64)


def inspect_star_counts(genes):
    count_values = pd.to_numeric(genes["unstranded"], errors="coerce")
    counts = count_values.to_numpy(dtype="float64", na_value=np.nan)
    finite = np.isfinite(counts)
    invalid = ~finite | (counts < 0) | (finite & (counts != np.floor(counts)))
    valid = not bool(invalid.any())

    frozen_mask = genes["gene_id"].isin(e_frozen_set).to_numpy()
    return {
        "unstranded_nonnegative_integer_vector": valid,
        "n_invalid_unstranded_gene_values": int(invalid.sum()),
        "unstranded_gene_count_total": float(counts.sum()) if valid else np.nan,
        "n_detected_star_genes": int((counts > 0).sum()) if valid else pd.NA,
        "n_detected_frozen_genes_exact": (
            int((counts[frozen_mask] > 0).sum()) if valid else pd.NA
        ),
    }


In [ ]:
# =============================================================================
# Combine STAR identity diagnostics with special-category tallies
# =============================================================================

def inspect_source_file(row):
    genes, special = read_star_source(row)
    features, exact_vector = inspect_star_features(genes)
    counts = inspect_star_counts(genes)
    special_counts = pd.to_numeric(special["unstranded"], errors="coerce")

    special_records = {}
    for label in E_STAR_SPECIAL:
        observed = special_counts.loc[special["gene_id"].eq(label)]
        special_records[f"star_{label.lower()}"] = (
            float(observed.iloc[0])
            if len(observed) == 1 and pd.notna(observed.iloc[0])
            else np.nan
        )

    record = {
        "case_uuid": row.case_uuid,
        "file_id": row.file_id,
        "sample_uuid": row.sample_uuid,
        "aliquot_uuid": row.aliquot_uuid,
        "filename": row.filename,
        "size_bytes": row.size_bytes,
        "provider_md5": row.provider_md5,
        "primary_site": row.primary_site,
        "disease_type": row.disease_type,
        "n_star_special_rows": len(special),
        **features,
        **counts,
        **special_records,
        "selection_performed": False,
        "technical_qc_interpretation": "EXACT_FEATURE_PRESENCE_NOT_SEQUENCING_QC",
    }
    return record, exact_vector


In [ ]:
# =============================================================================
# Construct or load exact gene-coverage evidence
# =============================================================================

if E_SCAN_PERFORMED:
    scan_records = []
    gene_presence = np.zeros(len(e_frozen_exact), dtype=np.int64)

    for index, row in enumerate(e_source.itertuples(index=False), start=1):
        record, exact_vector = inspect_source_file(row)
        scan_records.append(record)
        gene_presence += exact_vector

        if index % 200 == 0 or index == len(e_source):
            print(f"Source feature-coverage files: {index}/{len(e_source)}")

    e_file_coverage = pd.DataFrame(scan_records)
    e_gene_coverage = gene_frame[["gene_symbol", "tumor_gene_id"]].copy()
    e_gene_coverage["n_simple_files_exact_id_present"] = gene_presence
    e_gene_coverage["n_simple_files_inspected"] = len(e_file_coverage)
    e_gene_coverage["fraction_simple_files_exact_id_present"] = (
        e_gene_coverage["n_simple_files_exact_id_present"]
        / e_gene_coverage["n_simple_files_inspected"]
    )
else:
    e_file_coverage = pd.read_csv(E_FILE_PATH, dtype="string")
    e_gene_coverage = pd.read_csv(E_GENE_PATH, dtype="string")

# A completed raw-file read is retained even if a later consistency
# check fails. This receipt is provisional until all checks have passed.
if E_SCAN_PERFORMED:
    META.mkdir(parents=True, exist_ok=True)
    e_file_coverage.to_csv(E_FILE_PATH, index=False)
    e_gene_coverage.to_csv(E_GENE_PATH, index=False)
    pending_receipt = {
        "notebook": 800,
        "schema": E_AUDIT_SCHEMA,
        "inputs": E_INPUT_FINGERPRINTS,
        "source_identity_sha256": E_SOURCE_IDENTITY_SHA256,
        "n_source_files": 1082,
        "n_frozen_genes": 2389,
        "file_output_sha256": file_sha(E_FILE_PATH),
        "gene_output_sha256": file_sha(E_GENE_PATH),
        "verification_status": "PENDING_CROSSCHECK",
        "source_QC_certified": False,
        "case_selection_modified": False,
        "biological_replication_executed": False,
    }
    E_MANIFEST_PATH.write_text(
        json.dumps(pending_receipt, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

print("Source files:", len(e_file_coverage))
print("Frozen reference genes:", len(e_gene_coverage))


In [ ]:
# =============================================================================
# Validate feature-coverage data shape and reference order
# =============================================================================
e_required = source_identity + [
    "filename", "size_bytes", "provider_md5", "n_gene_rows",
    "n_duplicate_versionless_ids", "n_star_special_rows",
    "n_frozen_exact_present", "n_frozen_exact_missing",
    "n_frozen_versionless_diagnostic",
    "n_frozen_versionless_only_diagnostic",
    "unstranded_nonnegative_integer_vector",
    "n_invalid_unstranded_gene_values", "unstranded_gene_count_total",
    "n_detected_star_genes", "n_detected_frozen_genes_exact",
    "selection_performed", *["star_" + k.lower() for k in E_STAR_SPECIAL],
]
require_columns(e_file_coverage, e_required, "full E per-source feature audit")
require_columns(e_gene_coverage, [
    "gene_symbol", "tumor_gene_id", "n_simple_files_exact_id_present",
    "n_simple_files_inspected", "fraction_simple_files_exact_id_present",
], "full E frozen-gene presence")
check(len(e_file_coverage) == 1082 and
      e_file_coverage["case_uuid"].is_unique and
      e_file_coverage["file_id"].is_unique,
      "Full 1082-file E assessment lost cases or duplicated identities")
check(len(e_gene_coverage) == 2389 and
      e_gene_coverage["tumor_gene_id"].tolist() == e_frozen_exact.tolist(),
      "Full E per-gene records differ from frozen Phase 4 feature order")


In [ ]:
# =============================================================================
# Verify source case/file identities and GDC file size
# =============================================================================

source_check = e_file_coverage[source_identity + [
    "filename", "size_bytes", "provider_md5"
]].sort_values(source_identity, kind="stable").reset_index(drop=True)
expected_source_check = e_source[source_identity + [
    "filename", "size_bytes", "provider_md5"
]].sort_values(source_identity, kind="stable").reset_index(drop=True)
for col in source_identity + ["filename", "provider_md5"]:
    check(source_check[col].astype("string").equals(
          expected_source_check[col].astype("string")),
          f"E cached file identity differs from 113 G2: {col}")
observed_sizes = pd.to_numeric(
    source_check["size_bytes"], errors="raise"
).astype("Int64")
expected_sizes = pd.to_numeric(
    expected_source_check["size_bytes"], errors="raise"
).astype("Int64")
check(
    observed_sizes.notna().all()
    and expected_sizes.notna().all()
    and observed_sizes.eq(expected_sizes).fillna(False).all(),
    "Source-file sizes differ from the G2 file inventory",
)


In [ ]:
# =============================================================================
# Cross-check source count metrics with Notebook 112
# =============================================================================

e_vs_112 = e_file_coverage.merge(
    source_112, on=source_identity, how="inner", validate="one_to_one",
    suffixes=("_e", "_112"),
)
check(len(e_vs_112) == 1082, "E full audit and 112 source diagnostics not aligned")
shared_numeric = [
    "n_gene_rows", "n_duplicate_versionless_ids",
    "n_star_special_rows", "n_invalid_unstranded_gene_values",
    "unstranded_gene_count_total", "n_detected_star_genes",
    *["star_" + k.lower() for k in E_STAR_SPECIAL],
]
for col in shared_numeric:
    observed = pd.to_numeric(
        e_vs_112[f"{col}_e"], errors="raise"
    ).astype("Float64")
    expected = pd.to_numeric(
        e_vs_112[f"{col}_112"], errors="raise"
    ).astype("Float64")
    check(
        observed.notna().all()
        and expected.notna().all()
        and observed.eq(expected).fillna(False).all(),
        f"Source diagnostic disagreement for {col}",
    )
check(
    e_vs_112["unstranded_nonnegative_integer_vector_e"]
    .astype("string").str.lower().equals(
        e_vs_112["unstranded_nonnegative_integer_vector_112"]
        .astype("string").str.lower()
    ),
    "E STAR count-vector validity flags disagree with 112",
)
check(e_file_coverage["selection_performed"].astype("string").str.lower().eq(
    "false").all(), "E may not select files or set a QC threshold")


In [ ]:
# =============================================================================
# Verify exact reference gene coverage
# =============================================================================

# Exact gene-ID coverage is a source compatibility prerequisite, not a QC filter.
for col, expected in [
    ("n_frozen_exact_present", 2389),
    ("n_frozen_exact_missing", 0),
    ("n_frozen_versionless_only_diagnostic", 0),
    ("n_invalid_unstranded_gene_values", 0),
]:
    values = pd.to_numeric(e_file_coverage[col], errors="raise")
    check(values.eq(expected).all(),
          f"Expected feature regression check failed: {col}")
gene_presence = pd.to_numeric(
    e_gene_coverage["n_simple_files_exact_id_present"], errors="raise"
)
check(gene_presence.eq(1082).all(),
      "Frozen gene exact-presence across 1,082 files changed")
check(pd.to_numeric(
    e_gene_coverage["n_simple_files_inspected"], errors="raise"
).eq(1082).all(), "Gene-level denominator changed")
check(pd.to_numeric(
    e_file_coverage["n_detected_frozen_genes_exact"], errors="raise"
).between(0, 2389).all(), "Invalid frozen-gene detection count")
check(
    int(pd.to_numeric(e_file_coverage["n_frozen_exact_present"]).sum())
    == int(gene_presence.sum()),
    "Sum of per-file exact gene presence does not match per-gene coverage"
)


In [ ]:
# =============================================================================
# Summarize source-stratified feature coverage
# =============================================================================

e_full_context = (
    e_file_coverage.groupby(
        ["primary_site", "disease_type"], dropna=False
    ).agg(
        n_simple_source_files=("file_id", "nunique"),
        n_all_exact_frozen_genes=(
            "n_frozen_exact_missing",
            lambda x: int(pd.to_numeric(x).eq(0).sum())
        ),
        n_versionless_only_cases=(
            "n_frozen_versionless_only_diagnostic",
            lambda x: int(pd.to_numeric(x).gt(0).sum())
        ),
    ).reset_index()
)
check(e_full_context["n_simple_source_files"].sum() == 1082 and
      len(e_full_context) == 18, "E unharmonized source-context coverage drift")
check(set(cohort["file_id"]).issubset(set(e_file_coverage["file_id"])),
      "113 proposed cohort not contained in full E source universe")
display(e_file_coverage[[
    "n_gene_rows", "n_frozen_exact_present", "n_frozen_exact_missing",
    "n_frozen_versionless_only_diagnostic",
    "n_detected_frozen_genes_exact",
]].apply(pd.to_numeric, errors="raise").describe().T)
display(e_full_context.sort_values(
    "n_simple_source_files", ascending=False
).reset_index(drop=True))
print("Full-source source universe: 1,082/1,082 exact 2,389-feature coverage")
print("Phase 8 SOURCE INPUT compatibility verified; no eligibility change")


In [ ]:
# =============================================================================
# Certify reconciled feature-coverage evidence
# =============================================================================

e_manifest = read_json(E_MANIFEST_PATH)
check(
    e_manifest["schema"] == E_AUDIT_SCHEMA
    and e_manifest["inputs"] == E_INPUT_FINGERPRINTS
    and e_manifest["source_identity_sha256"] == E_SOURCE_IDENTITY_SHA256,
    "Feature-audit receipt does not match its source and reference inputs",
)
check(
    file_sha(E_FILE_PATH) == e_manifest["file_output_sha256"]
    and file_sha(E_GENE_PATH) == e_manifest["gene_output_sha256"],
    "Feature-audit derived files changed after inspection",
)
check(
    e_manifest.get("source_QC_certified") is False
    and e_manifest.get("biological_replication_executed") is False,
    "Feature compatibility cannot certify sequencing QC or replication",
)
e_manifest["verification_status"] = "VERIFIED"
e_manifest["source_feature_coverage_checks"] = {
    "all_1082_exact_2389_genes": True,
    "no_versionless_only_rescue_needed": True,
    "no_invalid_STAR_counts_detected": True,
}
E_MANIFEST_PATH.write_text(
    json.dumps(e_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("Frozen-feature audit:", e_manifest["verification_status"])
print("STAR files read in this run:", 1082 if E_SCAN_PERFORMED else 0)
print("Derived evidence:", E_MANIFEST_PATH.relative_to(ROOT))


## 6. Lineage support and potential confounding

Assess the observed workflow and source-context composition within the 721 proposed cases. Report upstream covariate-field coverage separately for the complete 1,082-case source universe.

Unknown purity, proliferation, batch or library characteristics must not be imputed. These measurements may limit the interpretability of cross-dataset comparisons.


In [ ]:
# =============================================================================
# Review 113 source covariates and actual proposed-cohort context support
# =============================================================================

g8_covariates = read_csv(G8_COVARIATES_NAME)
g8_workflows = read_csv(G8_WORKFLOW_NAME)
g8_contexts = read_csv(G8_CONTEXT_NAME)

require_columns(g8_covariates, [
    "covariate", "field_used", "n_present", "n_missing", "interpretation"
], "113 G8 covariates")
require_columns(g8_workflows, ["tcga_lineage", "workflow_type", "n_cases"],
                "113 G8 workflow support")
require_columns(g8_contexts, ["tcga_lineage", "g8_source_context", "n_cases"],
                "113 G8 source context support")
require_columns(ledger, ["workflow_type", "primary_site", "disease_type"],
                "113 case-level source context")

# The 113 G8 tables describe the complete G2 source candidates. For proposed-
# cohort support, summarize only the 721 exact accepted identities (without
# changing their membership or accessing expression outcomes).


In [ ]:
# =============================================================================
# Summarize workflow and source context within the cohort
# =============================================================================

selected_metadata = ledger.set_index("case_uuid").loc[
    cohort["case_uuid"].tolist()
].reset_index()
workflow_support = (
    selected_metadata.groupby(["tcga_lineage", "workflow_type"], dropna=False)
    .agg(n_cases=("case_uuid", "nunique"))
    .reset_index()
    .sort_values(["tcga_lineage", "workflow_type"])
)
context_support = (
    selected_metadata.groupby(
        ["tcga_lineage", "primary_site", "disease_type"], dropna=False
    )
    .agg(n_cases=("case_uuid", "nunique"))
    .reset_index()
    .sort_values(["tcga_lineage", "primary_site", "disease_type"])
)
check(int(workflow_support["n_cases"].sum()) == len(cohort),
      "Accepted-cohort workflow support does not reconcile")
check(int(context_support["n_cases"].sum()) == len(cohort),
      "Accepted-cohort context support does not reconcile")

print("G8 covariate field coverage among 1,082 G2 source candidates (not 721):")
display(g8_covariates)
print("Observed workflow distribution among 721 proposed source cases:")
display(workflow_support)
print("Observed source context distribution among 721 proposed source cases:")
display(context_support)
print("Adjustment feasibility / confounder identifiability: NOT ESTABLISHED")


## 7. Structural replication design: outstanding decisions

The intended evaluation concerns within-lineage correspondence of nonoverlapping gene blocks from fixed programs, relative to matched negative controls. A projected score or correlation with its constituent genes is not evidence of structural replication.

Normalization, gene-block partitioning, matched-null construction, uncertainty, multiplicity, lineage support and confounder sensitivity must be specified before inspecting external program results. No inferential analysis is performed here.


In [ ]:
# =============================================================================
# Report the prespecified testing universe without running biological analysis
# =============================================================================

n_programs = len(references["program_catalog"])
n_lineages = len(EXPECTED_LINEAGES)
design_summary = pd.DataFrame([
    {"component": "Frozen transcriptomic programs", "value": n_programs,
     "decision": "FIXED"},
    {"component": "Proposed source lineages", "value": n_lineages,
     "decision": "SOURCE_MAPPING_AVAILABLE"},
    {"component": "Potential program-by-lineage comparisons",
     "value": n_programs * n_lineages, "decision": "EVALUABILITY_PENDING"},
    {"component": "G6 external normalization and projection",
     "value": "not run", "decision": "DESIGN_REQUIRED"},
    {"component": "G7 statistic, matched controls and multiplicity",
     "value": "not run", "decision": "DESIGN_REQUIRED"},
    {"component": "G8 confounder sensitivity and support",
     "value": "not run", "decision": "DESIGN_REQUIRED"},
])
display(design_summary)
print("No lineage or program was selected on the basis of CPTAC expression outcomes")


## 8. Auditable readiness record

Persist the source-handoff and frozen-feature evidence fingerprints, lineage denominators and unresolved analytical requirements. Preserve the upstream source cohort and reference registry unchanged.

Readiness describes what can be evaluated next; it does not freeze the inferential cohort or authorize biological interpretation.


In [ ]:
# =============================================================================
# Write local handoff acceptance and lineage support; never freeze the cohort
# =============================================================================

META.mkdir(parents=True, exist_ok=True)
acceptance_path = META / "800_cptac3_handoff_acceptance.json"
support_path = META / "800_cptac3_lineage_support.csv"

acceptance = {
    "notebook": 800,
    "source_notebook": 113,
    "source_manifest_sha256": file_sha(MANIFEST_PATH),
    "source_readiness_sha256": file_sha(READINESS_PATH),
    "source_cohort_sha256": output_info[COHORT_NAME]["sha256"],
    "frozen_shared_gene_universe_sha256": (
        artifacts[reference_keys["shared_genes"]]["sha256"]
    ),
    "frozen_program_catalog_sha256": (
        artifacts[reference_keys["program_catalog"]]["sha256"]
    ),
    "frozen_program_weights_sha256": (
        artifacts[reference_keys["program_weights"]]["sha256"]
    ),
    "n_source_candidate_cases": len(ledger),
    "n_proposed_source_cases": len(cohort),
    "n_proposed_source_files": int(cohort["file_id"].nunique()),
    "n_frozen_genes": len(gene_ids),
    "n_frozen_programs": n_programs,
    "full_E_source_files_audited": len(e_file_coverage),
    "full_E_frozen_features_audited": len(e_gene_coverage),
    "full_E_exact_coverage_status": "1082_OF_1082_ALL_2389_EXACT_GENES",
    "full_E_source_input_manifest_sha256": file_sha(E_MANIFEST_PATH),
    "full_E_file_coverage_sha256": file_sha(E_FILE_PATH),
    "full_E_per_gene_coverage_sha256": file_sha(E_GENE_PATH),
    "full_E_distinct_from_721_G6_scope": True,
    "lineage_counts": dict(sorted(observed.items())),
    "g3": "COHORT_SUPPORTED_EXTERNAL_WITH_RESIDUAL_DONOR_UNCERTAINTY",
    "g5": "SOURCE_INTEGRITY_VERIFIED_READ_QC_NOT_CERTIFIED",
    "g6": "SOURCE_RAW_COUNTS_COMPATIBLE_ANALYTIC_TRANSFORMATION_OPEN",
    "g7": "STRUCTURAL_REPLICATION_DESIGN_OPEN",
    "g8": "CONFOUNDER_IDENTIFIABILITY_AND_SUPPORT_POLICY_OPEN",
    "source_handoff_verified": True,
    "cohort_frozen": False,
    "external_analysis_authorized": False,
    "biological_replication_executed": False,
    "verdict": "HANDOFF_VERIFIED_ANALYSIS_DESIGN_PENDING",
}


In [ ]:
# =============================================================================
# Write and verify readiness artifacts
# =============================================================================

lineage_support.to_csv(support_path, index=False)
acceptance_path.write_text(
    json.dumps(acceptance, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

check(read_json(acceptance_path) == acceptance,
      "800 acceptance record did not persist correctly")
check(pd.read_csv(support_path).shape[0] == n_lineages,
      "800 lineage-support report did not persist correctly")

print("800 handoff acceptance:", acceptance["verdict"])
print("Acceptance record:", acceptance_path.relative_to(ROOT),
      file_sha(acceptance_path))
print("Lineage support:", support_path.relative_to(ROOT),
      file_sha(support_path))
print("No registry change, no program scoring, no external-inference authorization")


## Interpretation

Successful execution establishes:

- Reconciled source handoff and case/file identities for 721 proposed cases.
- Full-source exact-feature coverage evidence for 1,082 structurally simple cases.
- Verified Phase 4 input identifiers and a lineage-aware description of available source metadata.

These are **technical prerequisites**, not external biological replication. The proposed cohort remains unfrozen, and inferential analysis requires prospective normalization, matched controls, uncertainty and confounder specifications.
